# RDF creation of tensile test data using TTO (PMDco 3.0)

This notebook reads tensile test data (S355 steel sheet) from a CSV source and transforms it into RDF (A-Box) using the **[Tensile Test Ontology (TTO)](https://github.com/materialdigital/tensile-test-ontology)** and the **[PMD Core Ontology (PMDco) 3.0.0](https://github.com/materialdigital/core-ontology/releases/tag/v3.0.0)**.

Precisely, data originating from tensile tests performed to an S355 steel sheet are read in to create an exemplary RDF data file that uses the [Tensile Test Ontology (TTO) version 3.0](https://github.com/materialdigital/tensile-test-ontology) as semantic backbone. Hence, the tensile test data available in 'classic' test formats is transformed to RDF data by creating triples using the RDFlib library and some helper functions. Therefore, values tokenized from the CSV are allocated to concepts of the TTO developed in the frame of the [PMD project](https://materialdigital.de/). Since the TTO is strongly based on the PMD Core Ontology - PMDco (namespace: [https://w3id.org/pmd/co](https://w3id.org/pmd/co)) - the PMDco is also parsed and semantic concepts this ontology is related to are strongly used, accordingly. 

The tensile test data can also be found in a [dataset published openly available on Zenodo](https://zenodo.org/record/6778336).

The RDF data is directly saved in a graph. Subsequently, the graph is serialized and saved to a TTL and an RDF file. Furthermore, SPARQL queries may be performed for consistency checks.
Hence, all in all, this script is used to create a knowledge graph (A-Box) using vocabulary of several ontologies and tensile test data originating from real tests performed on an S355 steel in accordance with the corresponding test standard [ISO 6892-1:2019-11](https://www.beuth.de/de/norm/iso-6892-1/316885984).

**Semantics in accordance with PMDco 3.0 patterns:**
- Scalar Value Measurement Pattern (PMD_0000022, PMD_0000023, PMD_0060000, PMD_0060001, PMD_0000006)
- Process-Attribute modeling for process characteristics
- Sub-process skeleton (pre-measuring, mounting, testing, unmounting, post-measuring, analysis) with temporal ordering and part-of relations
    - Instruments linked to both parent process and appropriate sub-processes

**Extended metadata:**
- Devices: tensile testing machine, extensometer, load cell, micrometer gauge, caliper (incl. name, type, serial number, standard, manufacturer)
- Organization: institute (name + address/location)
- Standard (Norm) → Plan Specification (tensile test method plan) realized by the process
- Project → Project entity with Identifier
- Walzrichtung → Material quality with categorical value

**Example semantic representations and debugging:**
- CSVW table description for the input CSV, linked to the tensile test process as an information input
- Debug/SELECT blocks to list violating IRIs if an ASK check fails


Notebook structure mirrors the legacy notebook: install → helpers → data → graph → main mapping (→ checking / debugging) → serialization.

# Installing and Importing relevant Python packages

In [1]:
%%capture
%pip install --upgrade pip
%pip install rdflib pandas

In [2]:
# Imports
from rdflib import Graph, Literal, URIRef, BNode, Namespace
from rdflib.namespace import RDF, RDFS, XSD, OWL
import pandas as pd
from datetime import datetime

# Helper functions

In [ ]:
# Turtle-style abbreviation for rdf:type
a = RDF.type

# Helper method 'add' used to write triples to an RDF graph
def add(s,p,o):

    # in this case p is "ObjectProperty"
    if o.find('http://')==0 or o.find('https://')==0:
        g.add( (URIRef(s), URIRef(p), URIRef(o)) )

    # in this case p is "DatatypeProperty"
    else:
        # if we can parse o as Float, just set the datatype
        try:
            g.add( (URIRef(s), URIRef(p), Literal(float(o), datatype=XSD.float) ))
        except:
            g.add( (URIRef(s), URIRef(p), Literal(o) ))

def as_float(v):
    # Robust conversion: handles numbers, strings, German decimal commas
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).strip()
    if s == '' or s.lower() in {'nan', 'none'}:
        return None
    s = s.replace(',', '.')
    try:
        return float(s)
    except Exception:
        return None


def add_identifier(
    g, base_iri, thing_iri, value, *,
    id_suffix, identifier_class, value_spec_class,
    pmd_has_value, pmd_has_value_spec, iao_denotes, label=None
):
    # PMD identifier + OBI value specification patterns
    if value is None or str(value).strip() == '' or str(value).lower() == 'nan':
        return None
    ident_iri = URIRef(f"{base_iri}_{id_suffix}")
    val_iri = URIRef(f"{base_iri}_{id_suffix}_value")
    g.add((ident_iri, a, identifier_class))
    g.add((ident_iri, iao_denotes, thing_iri))
    g.add((val_iri, a, value_spec_class))
    g.add((val_iri, pmd_has_value, Literal(str(value), datatype=XSD.string)))
    g.add((ident_iri, pmd_has_value_spec, val_iri))
    if label:
        g.add((ident_iri, RDFS.label, Literal(label, datatype=XSD.string)))
    return ident_iri


def add_scalar_value_measurement(
    g, base_iri, *,
    measured_entity_class, measurement_name, numeric_value, unit_iri,
    bearer_iri, bearer_predicate, process_iri, result_iri,
    scalar_value_spec_class, scalar_measurement_datum_class,
    pmd_has_value, pmd_has_unit, pmd_has_value_spec, pmd_specifies_value_of,
    obi_has_specified_output, bfo_has_part, iao_is_about, iao_is_quality_measured_as
):
    # Create measured entity + scalar value specification + scalar measurement datum (PMDco pattern)
    v = as_float(numeric_value)
    if v is None:
        return None

    q_iri = URIRef(f"{base_iri}_{measurement_name}")
    g.add((q_iri, a, measured_entity_class))
    g.add((bearer_iri, bearer_predicate, q_iri))

    svs_iri = URIRef(f"{base_iri}_{measurement_name}_svs")
    g.add((svs_iri, a, scalar_value_spec_class))
    g.add((svs_iri, pmd_specifies_value_of, q_iri))
    g.add((svs_iri, pmd_has_value, Literal(v, datatype=XSD.float)))
    if unit_iri is not None:
        g.add((svs_iri, pmd_has_unit, unit_iri))
    g.add((svs_iri, iao_is_about, q_iri))

    smd_iri = URIRef(f"{base_iri}_{measurement_name}_smd")
    g.add((smd_iri, a, scalar_measurement_datum_class))
    g.add((smd_iri, pmd_has_value_spec, svs_iri))
    g.add((smd_iri, iao_is_about, q_iri))
    g.add((q_iri, iao_is_quality_measured_as, smd_iri))
    
    # process output + aggregate into tensile test result
    g.add((process_iri, obi_has_specified_output, smd_iri))
    g.add((result_iri, bfo_has_part, smd_iri))
    return smd_iri

# Device helper (device + function + optional metadata + manufacturer org)

def add_device(
    g, base_iri, *,
    device_denotation, device_class, function_class,
    process_iri, participant_predicate, bearer_of_predicate, realized_in_predicate,
    name=None, manufacturer_name=None, device_type=None, serial_number=None, standard=None,
    identifier_class=None, value_spec_class=None, pmd_has_value=None, pmd_has_value_spec=None, iao_denotes=None
):
    dev_iri = URIRef(f"{base_iri}_{device_denotation}")
    func_iri = URIRef(f"{base_iri}_{device_denotation}_function")

    g.add((dev_iri, a, device_class))
    g.add((func_iri, a, function_class))

    g.add((dev_iri, bearer_of_predicate, func_iri))
    g.add((func_iri, realized_in_predicate, process_iri))
    g.add((process_iri, participant_predicate, dev_iri))

    def _add_id(val, suffix, lbl):
        if val is None or str(val).strip() == '' or str(val).lower() == 'nan':
            return
        add_identifier(
            g, base_iri=str(dev_iri), thing_iri=dev_iri, value=val,
            id_suffix=suffix, identifier_class=identifier_class,
            value_spec_class=value_spec_class, pmd_has_value=pmd_has_value,
            pmd_has_value_spec=pmd_has_value_spec, iao_denotes=iao_denotes,
            label=lbl
        )

    _add_id(name, 'name', f"{device_denotation} name")
    _add_id(device_type, 'type', f"{device_denotation} type")
    _add_id(serial_number, 'serial_number', f"{device_denotation} serial number")
    _add_id(standard, 'standard', f"{device_denotation} standard")

    if manufacturer_name is not None and str(manufacturer_name).strip() != '' and str(manufacturer_name).lower() != 'nan':
        manuf_iri = URIRef(f"{base_iri}_{device_denotation}_manufacturer")
        g.add((manuf_iri, a, obo.OBI_0000245))
        g.add((process_iri, participant_predicate, manuf_iri))
        add_identifier(
            g, base_iri=str(manuf_iri), thing_iri=manuf_iri, value=manufacturer_name,
            id_suffix='name', identifier_class=identifier_class,
            value_spec_class=value_spec_class, pmd_has_value=pmd_has_value,
            pmd_has_value_spec=pmd_has_value_spec, iao_denotes=iao_denotes,
            label=f"{device_denotation} manufacturer name"
        )

    return dev_iri

# Organization helper (institute + address/location)

def add_institution_and_location(
    g, base_iri, *,
    process_iri, participant_predicate,
    organization_class, institute_name=None, address=None, location=None,
    identifier_class=None, value_spec_class=None,
    pmd_has_value=None, pmd_has_value_spec=None, iao_denotes=None
):
    inst_iri = URIRef(f"{base_iri}_institution")
    g.add((inst_iri, a, organization_class))
    g.add((process_iri, participant_predicate, inst_iri))

    if institute_name is not None and str(institute_name).strip() != '' and str(institute_name).lower() != 'nan':
        add_identifier(
            g, base_iri=str(inst_iri), thing_iri=inst_iri, value=institute_name,
            id_suffix='name', identifier_class=identifier_class,
            value_spec_class=value_spec_class, pmd_has_value=pmd_has_value,
            pmd_has_value_spec=pmd_has_value_spec, iao_denotes=iao_denotes,
            label='institution name'
        )

    parts = []
    if address is not None and str(address).strip() != '' and str(address).lower() != 'nan':
        parts.append(str(address).strip())
    if location is not None and str(location).strip() != '' and str(location).lower() != 'nan':
        parts.append(str(location).strip())

    if parts:
        addr_full = ' '.join(parts)
        add_identifier(
            g, base_iri=str(inst_iri), thing_iri=inst_iri, value=addr_full,
            id_suffix='address', identifier_class=identifier_class,
            value_spec_class=value_spec_class, pmd_has_value=pmd_has_value,
            pmd_has_value_spec=pmd_has_value_spec, iao_denotes=iao_denotes,
            label='institution address'
        )

    return inst_iri


# Read in tensile test data

In [4]:
link_data = "https://raw.githubusercontent.com/materialdigital/tensile-test-ontology/refs/heads/main/tensile_test_data/S355_data.csv"
data = pd.read_csv(link_data, sep=';')
print('Rows, Cols:', data.shape)
data.head()

Rows, Cols: (10, 57)


,process,testpiece,d1 / a1,d2 / a2,d3 / a3,b1,b2,b3,s1,s2,...,Dehngeschwindigkeit,Umschaltpunkt,Extensometer-Nenngerätemesslänger,Kraftmessdose-Kraftbereich,Project,Norm,Walzrichtung,Ort,Institut,Adresse
0,pmdao-tto-tt-S355-1,Zx1,5.993,5.987,5.992,20.136,20.145,20.131,120.675,120.608,...,0.00025,1.6,50,100,PMDao-onto_1,DIN EN ISO 6892-1,in rolling direction,Berlin,Bundesanstalt für Materialforschung und -prüfu...,Unter den Eichen 87. 12205 Berlin
1,pmdao-tto-tt-S355-2,Zx2,5.991,5.991,5.992,20.114,20.101,20.094,120.503,120.425,...,0.00025,3.8,50,100,PMDao-onto_1,DIN EN ISO 6892-1,in rolling direction,Berlin,Bundesanstalt für Materialforschung und -prüfu...,Unter den Eichen 87. 12205 Berlin
2,pmdao-tto-tt-S355-3,Zx3,5.991,5.994,5.992,20.084,20.092,20.099,120.323,120.431,...,0.00025,3.8,50,100,PMDao-onto_1,DIN EN ISO 6892-1,in rolling direction,Berlin,Bundesanstalt für Materialforschung und -prüfu...,Unter den Eichen 87. 12205 Berlin
3,pmdao-tto-tt-S355-4,Zx4,5.982,5.978,5.977,20.076,20.072,45.127,120.095,119.990,...,0.00025,3.8,50,100,PMDao-onto_1,DIN EN ISO 6892-1,in rolling direction,Berlin,Bundesanstalt für Materialforschung und -prüfu...,Unter den Eichen 87. 12205 Berlin
4,pmdao-tto-tt-S355-5,Zy1,6.000,5.992,5.993,20.129,20.133,20.128,120.774,120.637,...,0.00025,3.8,50,100,PMDao-onto_1,DIN EN ISO 6892-1,perpendicular to rolling direction,Berlin,Bundesanstalt für Materialforschung und -prüfu...,Unter den Eichen 87. 12205 Berlin


# Graph creation using RDFlib

In [5]:
g = Graph()

tto = Namespace("https://w3id.org/pmd/tto/")
pmdco = Namespace("https://w3id.org/pmd/co/")
obo = Namespace("http://purl.obolibrary.org/obo/")
unit = Namespace("http://qudt.org/vocab/unit/")
dct = Namespace("http://purl.org/dc/terms/")
dc = Namespace("http://purl.org/dc/elements/1.1/")
time = Namespace("http://www.w3.org/2006/time#")
csvw = Namespace("http://www.w3.org/ns/csvw#")

for prefix, ns in [("tto", tto), ("pmdco", pmdco), ("obo", obo), ("unit", unit), ("dct", dct), ("dc", dc), ("time", time), ("csvw", csvw)]:
    g.bind(prefix, ns)

abox = Namespace("https://w3id.org/pmd/demodata/tensiletest_S355/")
g.bind("abox", abox)

onto = URIRef(abox)
g.add((onto, a, OWL.Ontology))
# Import TTO (imports PMDco)
g.add((onto, OWL.imports, URIRef("https://w3id.org/pmd/tto/")))

# Minimal metadata of graph/ABox/ontology file created
now = datetime.utcnow().strftime('%Y-%m-%d')
g.add((onto, dc.title, Literal("TTO/PMDco 3.0 A-Box mapping example of S355 steel tensile test data", datatype=XSD.string)))
g.add((onto, dct.created, Literal(now, datatype=XSD.date)))


<Graph identifier=N0c0511c54db842ce83cc38325c3fa653 (<class 'rdflib.graph.Graph'>)>

# CSVW table description (auto-generated from the CSV header)

In [6]:
# Create a CSVW description of the input table and keep it reusable for all processes

table_iri = abox.S355_data_csvw_table
schema_iri = abox.S355_data_csvw_schema

g.add((table_iri, a, csvw.Table))
# also type as IAO:dataset (information content entity)
g.add((table_iri, a, obo.IAO_0000100))
g.add((table_iri, csvw.url, Literal(link_data, datatype=XSD.anyURI)))
g.add((table_iri, csvw.headerCount, Literal(1, datatype=XSD.nonNegativeInteger)))
g.add((table_iri, csvw.tableSchema, schema_iri))
g.add((schema_iri, a, csvw.Schema))

# Create column descriptors
for col in data.columns:
    col_iri = URIRef(str(table_iri) + '/col/' + str(col).replace(' ', '_').replace('/', '_').replace(',', '_'))
    g.add((col_iri, a, csvw.Column))
    g.add((col_iri, csvw.name, Literal(str(col), datatype=XSD.string)))
    # simple datatype mapping
    dtype = str(data[col].dtype)
    if dtype.startswith('float') or dtype.startswith('int'):
        g.add((col_iri, csvw.datatype, Literal('decimal', datatype=XSD.string)))
    else:
        g.add((col_iri, csvw.datatype, Literal('string', datatype=XSD.string)))
    g.add((schema_iri, csvw.column, col_iri))

print('CSVW columns described:', len(list(data.columns)))


CSVW columns described: 57


# Main Data Mapping (A-Box creation)

In [ ]:
# --- Pattern IRIs ---
PMD_SVS = obo.OBI_0001931
PMD_SMD = obo.IAO_0000109
PMD_HAS_VALUE = pmdco.PMD_0000006
PMD_HAS_UNIT = obo.IAO_0000039
PMD_HAS_VALUE_SPEC = obo.OBI_0001938
PMD_SPECIFIES_VALUE_OF = obo.OBI_0001927

OBI_HAS_SPECIFIED_INPUT = obo.OBI_0000293
OBI_HAS_SPECIFIED_OUTPUT = obo.OBI_0000299
RO_HAS_QUALITY = obo.RO_0000086
RO_HAS_PARTICIPANT = obo.RO_0000057

BFO_HAS_PART = obo.BFO_0000051
BFO_OCCURRENT_PART_OF = obo.BFO_0000132
BFO_HAS_OCCURRENT_PART = obo.BFO_0000117
BFO_PRECEDED_BY = obo.BFO_0000062
BFO_PRECEDES = obo.BFO_0000063
BFO_REALIZES = obo.BFO_0000055

IAO_IS_ABOUT = obo.IAO_0000136
IAO_IS_QUALITY_MEASURED_AS = obo.IAO_0000417

PMD_HAS_PROCESS_ATTRIBUTE = pmdco.PMD_0000009

# Core classes
TENSILE_TEST_PROCESS = pmdco.PMD_0000974
TEST_PIECE_ROLE = pmdco.PMD_0000975
TENSILE_TEST_RESULT = tto.TTO_0000008
BROKEN_TEST_PIECE_ROLE = tto.TTO_0000006

STEEL = pmdco.PMD_0020096
ELASTIC_MODULUS = pmdco.PMD_0000618
ENV_TEMPERATURE = pmdco.PMD_0000967

# Ensure units are typed as IAO:measurement unit label
for u in [unit.MilliM, unit.MilliM2, unit.SEC, unit.KiloN, unit.MegaPa, unit.GigaPa, unit.PERCENT, unit.DEG_C]:
    g.add((u, a, obo.IAO_0000003))

# track which CSV columns we actually use in row.get(...)
USED_COLS = {
  'process','testpiece','d1 / a1','d2 / a2','d3 / a3','b1','b2','b3','s1','s2','s3','S0','L0','Lu','au','bu','Su',
  'Rp0,2','ReH','Fm','Rm','E','A','Z',
  'Dehngeschwindigkeit','Umschaltpunkt','Umgebungstemperatur',
  'Operator','date',
  'Ort','Institut','Adresse',
  'Project','Norm','Walzrichtung',
  'Prüfmaschinen-Name','Prüfmaschinenhersteller','Prüfmaschinen-Typ','Prüfmaschinen-Seriennummer','Prüfmaschinen-Standard',
  'Extensometer','Extensometer-Seriennummer','Extensometer-Standard',
  'Messschieber-Hersteller','Messschieber-Seriennummer',
  'Mikrometerschraube-Hersteller',
  'Extensometer-Nenngerätemesslänger','F0,2', 'FeL', 'ReL', 'FeH', 'du1', 'du2', 'Werkstoff', 'Probenform', 'Note', 'Kraftmessdose-Kraftbereich'
}

for _, row in data.iterrows():
    process_id = str(row.get('process'))
    testpiece_id = str(row.get('testpiece'))
    exp_iri = URIRef(abox + process_id)

    # Parent process
    process_iri = URIRef(str(exp_iri) + '_process')
    g.add((process_iri, a, TENSILE_TEST_PROCESS))

    # Attach CSVW table description as an information output of the process
    g.add((process_iri, OBI_HAS_SPECIFIED_OUTPUT, table_iri))

    add_identifier(
        g, base_iri=str(exp_iri), thing_iri=process_iri, value=process_id,
        id_suffix='process_identifier', identifier_class=obo.IAO_0020000,
        value_spec_class=obo.OBI_0001933, pmd_has_value=PMD_HAS_VALUE,
        pmd_has_value_spec=PMD_HAS_VALUE_SPEC, iao_denotes=obo.IAO_0000219,
        label='Process identifier'
    )

    # Organization
    add_institution_and_location(
        g, base_iri=str(exp_iri), process_iri=process_iri,
        participant_predicate=RO_HAS_PARTICIPANT,
        organization_class=obo.OBI_0000245,
        institute_name=row.get('Institut'),
        address=row.get('Adresse'),
        location=row.get('Ort'),
        identifier_class=obo.IAO_0020000,
        value_spec_class=obo.OBI_0001933,
        pmd_has_value=PMD_HAS_VALUE,
        pmd_has_value_spec=PMD_HAS_VALUE_SPEC,
        iao_denotes=obo.IAO_0000219
    )

    # Devices (also keep references for assigning to subprocesses)
    tensile_machine = add_device(
        g, base_iri=str(exp_iri), device_denotation='tensile_testing_machine',
        device_class=pmdco.PMD_0000973, function_class=pmdco.PMD_0000976,
        process_iri=process_iri, participant_predicate=RO_HAS_PARTICIPANT,
        bearer_of_predicate=obo.BFO_0000196, realized_in_predicate=obo.BFO_0000054,
        name=row.get('Prüfmaschinen-Name'),
        manufacturer_name=row.get('Prüfmaschinenhersteller'),
        device_type=row.get('Prüfmaschinen-Typ'),
        serial_number=row.get('Prüfmaschinen-Seriennummer'),
        standard=row.get('Prüfmaschinen-Standard'),
        identifier_class=obo.IAO_0020000, value_spec_class=obo.OBI_0001933,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_value_spec=PMD_HAS_VALUE_SPEC,
        iao_denotes=obo.IAO_0000219
    )

    extensometer = add_device(
        g, base_iri=str(exp_iri), device_denotation='extensometer',
        device_class=pmdco.PMD_0000636, function_class=pmdco.PMD_0000610,
        process_iri=process_iri, participant_predicate=RO_HAS_PARTICIPANT,
        bearer_of_predicate=obo.BFO_0000196, realized_in_predicate=obo.BFO_0000054,
        name=row.get('Extensometer'),
        serial_number=row.get('Extensometer-Seriennummer'),
        standard=row.get('Extensometer-Standard'),
        identifier_class=obo.IAO_0020000, value_spec_class=obo.OBI_0001933,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_value_spec=PMD_HAS_VALUE_SPEC,
        iao_denotes=obo.IAO_0000219
    )

    load_cell = add_device(
        g, base_iri=str(exp_iri), device_denotation='load_cell',
        device_class=pmdco.PMD_0000820, function_class=pmdco.PMD_0000644,
        process_iri=process_iri, participant_predicate=RO_HAS_PARTICIPANT,
        bearer_of_predicate=obo.BFO_0000196, realized_in_predicate=obo.BFO_0000054,
        identifier_class=obo.IAO_0020000, value_spec_class=obo.OBI_0001933,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_value_spec=PMD_HAS_VALUE_SPEC,
        iao_denotes=obo.IAO_0000219
    )

    micrometer = add_device(
        g, base_iri=str(exp_iri), device_denotation='micrometer_gauge',
        device_class=pmdco.PMD_0000854, function_class=pmdco.PMD_0000610,
        process_iri=process_iri, participant_predicate=RO_HAS_PARTICIPANT,
        bearer_of_predicate=obo.BFO_0000196, realized_in_predicate=obo.BFO_0000054,
        manufacturer_name=row.get('Mikrometerschraube-Hersteller'),
        identifier_class=obo.IAO_0020000, value_spec_class=obo.OBI_0001933,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_value_spec=PMD_HAS_VALUE_SPEC,
        iao_denotes=obo.IAO_0000219
    )

    caliper = add_device(
        g, base_iri=str(exp_iri), device_denotation='caliper',
        device_class=pmdco.PMD_0000544, function_class=pmdco.PMD_0000610,
        process_iri=process_iri, participant_predicate=RO_HAS_PARTICIPANT,
        bearer_of_predicate=obo.BFO_0000196, realized_in_predicate=obo.BFO_0000054,
        manufacturer_name=row.get('Messschieber-Hersteller'),
        serial_number=row.get('Messschieber-Seriennummer'),
        identifier_class=obo.IAO_0020000, value_spec_class=obo.OBI_0001933,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_value_spec=PMD_HAS_VALUE_SPEC,
        iao_denotes=obo.IAO_0000219
    )

    # --- Subprocesses ---
    pre_meas = URIRef(str(exp_iri) + '_process_preMeasuring')
    mounting = URIRef(str(exp_iri) + '_process_mounting')
    testing = URIRef(str(exp_iri) + '_process_testing')
    unmounting = URIRef(str(exp_iri) + '_process_unmounting')
    post_meas = URIRef(str(exp_iri) + '_process_postMeasuring')    
    analysis = URIRef(str(exp_iri) + '_process_analysis')

    subprocs = [
        (pre_meas, 'pre-measuring'),
        (mounting, 'mounting'),
        (testing, 'testing'),
        (unmounting, 'unmounting'),
        (post_meas, 'post-measuring'),        
        (analysis, 'analysis')
    ]

    for sp, lbl in subprocs:
        g.add((sp, a, obo.BFO_0000015))
        g.add((sp, RDFS.label, Literal(lbl, datatype=XSD.string)))
        g.add((sp, BFO_OCCURRENT_PART_OF, process_iri))
        g.add((process_iri, BFO_HAS_OCCURRENT_PART, sp))

    # temporal ordering (precedes / preceded_by)
    chain = [pre_meas, mounting, testing, unmounting, post_meas, analysis]
    for a1, a2 in zip(chain, chain[1:]):
        g.add((a2, BFO_PRECEDED_BY, a1))
        g.add((a1, BFO_PRECEDES, a2))

    # assign instruments to suitable subprocesses + realize function in those subprocesses
    # caliper + micrometer used in pre- and post-measuring
    for sp in [pre_meas, post_meas]:
        g.add((sp, RO_HAS_PARTICIPANT, caliper))
        g.add((sp, RO_HAS_PARTICIPANT, micrometer))
        g.add((URIRef(str(exp_iri) + '_caliper_function'), obo.BFO_0000054, sp))
        g.add((URIRef(str(exp_iri) + '_micrometer_gauge_function'), obo.BFO_0000054, sp))

    # tensile machine, extensometer, load cell used in testing
    for dev, func_suffix in [(tensile_machine, 'tensile_testing_machine_function'), (extensometer, 'extensometer_function'), (load_cell, 'load_cell_function')]:
        g.add((testing, RO_HAS_PARTICIPANT, dev))
        g.add((URIRef(str(exp_iri) + '_' + func_suffix), obo.BFO_0000054, testing))

    # Standard (Norm) as Plan Specification
    standard_val = row.get('Norm')
    if standard_val is not None and str(standard_val).strip() != '' and str(standard_val).lower() != 'nan':
        method_iri = URIRef(str(exp_iri) + '_tensile_test_method')
        g.add((method_iri, a, tto.TTO_0000054))
        g.add((process_iri, BFO_REALIZES, method_iri))
        add_identifier(
            g, base_iri=str(method_iri), thing_iri=method_iri, value=standard_val,
            id_suffix='standard', identifier_class=obo.IAO_0020000,
            value_spec_class=obo.OBI_0001933, pmd_has_value=PMD_HAS_VALUE,
            pmd_has_value_spec=PMD_HAS_VALUE_SPEC, iao_denotes=obo.IAO_0000219,
            label='tensile test method standard'
        )

    # Project entity with Identifier
    project_val = row.get('Project')
    if project_val is not None and str(project_val).strip() != '' and str(project_val).lower() != 'nan':
        project_iri = URIRef(str(exp_iri) + '_project')
        g.add((project_iri, a, pmdco.PMD_0000909))
        g.add((project_iri, BFO_HAS_OCCURRENT_PART, process_iri))
        add_identifier(
            g, base_iri=str(project_iri), thing_iri=project_iri, value=project_val,
            id_suffix='identifier', identifier_class=obo.IAO_0020000,
            value_spec_class=obo.OBI_0001933, pmd_has_value=PMD_HAS_VALUE,
            pmd_has_value_spec=PMD_HAS_VALUE_SPEC, iao_denotes=obo.IAO_0000219,
            label='project identifier'
        )

    # Result
    result_iri = URIRef(str(exp_iri) + '_result')
    g.add((result_iri, a, TENSILE_TEST_RESULT))
    g.add((process_iri, OBI_HAS_SPECIFIED_OUTPUT, result_iri))

    # Test piece
    testpiece_iri = URIRef(abox + f'testpiece/{testpiece_id}')
    g.add((testpiece_iri, a, obo.BFO_0000030))
    add_identifier(
        g, base_iri=str(testpiece_iri), thing_iri=testpiece_iri, value=testpiece_id,
        id_suffix='testpiece_identifier', identifier_class=obo.IAO_0020000,
        value_spec_class=obo.OBI_0001933, pmd_has_value=PMD_HAS_VALUE,
        pmd_has_value_spec=PMD_HAS_VALUE_SPEC, iao_denotes=obo.IAO_0000219,
        label='Test piece identifier'
    )

    tp_role = URIRef(str(testpiece_iri) + '_role')
    g.add((tp_role, a, TEST_PIECE_ROLE))
    g.add((testpiece_iri, obo.BFO_0000196, tp_role))
    g.add((process_iri, OBI_HAS_SPECIFIED_INPUT, testpiece_iri))

    # broken outputs
    tp_after_1 = URIRef(str(testpiece_iri) + '_afterTest_1')
    tp_after_2 = URIRef(str(testpiece_iri) + '_afterTest_2')
    for tp in [tp_after_1, tp_after_2]:
        g.add((tp, a, obo.BFO_0000030))
        g.add((process_iri, OBI_HAS_SPECIFIED_OUTPUT, tp))
        r = URIRef(str(tp) + '_role')
        g.add((r, a, BROKEN_TEST_PIECE_ROLE))
        g.add((tp, obo.BFO_0000196, r))

    # Material
    material_iri = URIRef(str(testpiece_iri) + '_material')
    g.add((material_iri, a, STEEL))
    g.add((testpiece_iri, pmdco.PMD_0020005, material_iri))
    
    # ------------------------------------------------------------------
    # Explicit material denotation ("Werkstoff", string) as identifier of the material
    # Column: Werkstoff
    # ------------------------------------------------------------------
    werkstoff_val = row.get('Werkstoff')
    if werkstoff_val is not None and str(werkstoff_val).strip() != '' and str(werkstoff_val).lower() != 'nan':
        add_identifier(
            g, base_iri=str(material_iri), thing_iri=material_iri, value=werkstoff_val,
            id_suffix='material_identifier', identifier_class=obo.IAO_0020000,
            value_spec_class=obo.OBI_0001933, pmd_has_value=PMD_HAS_VALUE,
            pmd_has_value_spec=PMD_HAS_VALUE_SPEC, iao_denotes=obo.IAO_0000219,
            label='material identifier (Werkstoff)'
        )

    # Walzrichtung as material quality
    roll_val = row.get('Walzrichtung')
    if roll_val is not None and str(roll_val).strip() != '' and str(roll_val).lower() != 'nan':
        roll_q = URIRef(str(material_iri) + '_rolling_direction')
        g.add((roll_q, a, obo.BFO_0000019))
        g.add((material_iri, RO_HAS_QUALITY, roll_q))
        roll_vs = URIRef(str(roll_q) + '_value_spec')
        g.add((roll_vs, a, obo.OBI_0001933))
        g.add((roll_vs, PMD_HAS_VALUE, Literal(str(roll_val), datatype=XSD.string)))
        g.add((roll_q, PMD_HAS_VALUE_SPEC, roll_vs))
        g.add((roll_vs, IAO_IS_ABOUT, material_iri))
        
    # ------------------------------------------------------------------
    # Test piece geometry shape as pmdco:PMD_0050156 (geometry shape)
    # Column: Probenform
    # ------------------------------------------------------------------
    probenform_val = row.get('Probenform')
    if probenform_val is not None and str(probenform_val).strip() != '' and str(probenform_val).lower() != 'nan':
        shape_q = URIRef(str(testpiece_iri) + '_geometry_shape')
        g.add((shape_q, a, pmdco.PMD_0050156))          # pmd:geometry shape (quality class)
        g.add((testpiece_iri, RO_HAS_QUALITY, shape_q)) # attach as quality of the test piece

        shape_vs = URIRef(str(shape_q) + '_value_spec')
        g.add((shape_vs, a, obo.OBI_0001933))
        g.add((shape_vs, PMD_HAS_VALUE, Literal(str(probenform_val), datatype=XSD.string)))
        g.add((shape_q, PMD_HAS_VALUE_SPEC, shape_vs))
        g.add((shape_vs, IAO_IS_ABOUT, testpiece_iri))

        
        
    # ------------------------------------------------------------------
    # Forces / stresses (only if values exist)
    # Columns: F0,2 ; FeL ; FeH ; ReL
    # TTO classes:
    #  - F0,2 -> tto:TTO_0000042 (force at proof strength plastic extension f02)
    #  - FeL  -> tto:TTO_0000010 (force at lower yield strength)
    #  - FeH  -> tto:TTO_0000021 (force at upper yield strength)
    #  - ReL  -> tto:TTO_0000022 (lower yield strength)
    # ------------------------------------------------------------------

    # F0,2 (force at proof strength plastic extension f02) – assume unit: kN; add only if numeric value present
    add_scalar_value_measurement(
        g, str(exp_iri), measured_entity_class=tto.TTO_0000042,
        measurement_name='force_at_proof_strength_plastic_extension_Fp0_2',
        numeric_value=row.get('F0,2'),
        unit_iri=unit.KiloN,
        bearer_iri=material_iri, bearer_predicate=RO_HAS_QUALITY,
        process_iri=process_iri, result_iri=result_iri,
        scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
        pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
        obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
        iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
    )

    # FeL (force at lower yield strength) – assume unit: kN
    add_scalar_value_measurement(
        g, str(exp_iri), measured_entity_class=tto.TTO_0000010,
        measurement_name='force_at_lower_yield_strength_FeL',
        numeric_value=row.get('FeL'),
        unit_iri=unit.KiloN,
        bearer_iri=material_iri, bearer_predicate=RO_HAS_QUALITY,
        process_iri=process_iri, result_iri=result_iri,
        scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
        pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
        obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
        iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
    )

    # FeH (force at upper yield strength) – assume unit: kN
    add_scalar_value_measurement(
        g, str(exp_iri), measured_entity_class=tto.TTO_0000021,
        measurement_name='force_at_upper_yield_strength_FeH',
        numeric_value=row.get('FeH'),
        unit_iri=unit.KiloN,
        bearer_iri=material_iri, bearer_predicate=RO_HAS_QUALITY,
        process_iri=process_iri, result_iri=result_iri,
        scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
        pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
        obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
        iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
    )

    # ReL (lower yield strength) – unit: MPa
    add_scalar_value_measurement(
        g, str(exp_iri), measured_entity_class=tto.TTO_0000022,
        measurement_name='lower_yield_strength_ReL',
        numeric_value=row.get('ReL'),
        unit_iri=unit.MegaPa,
        bearer_iri=material_iri, bearer_predicate=RO_HAS_QUALITY,
        process_iri=process_iri, result_iri=result_iri,
        scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
        pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
        obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
        iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
    )

    # Device extra raw fields (if present)
    # extensometer nominal gauge length
    ext_nom = row.get('Extensometer-Nenngerätemesslänger')
    if ext_nom is not None and str(ext_nom).strip() != '' and str(ext_nom).lower() != 'nan':
        add_scalar_value_measurement(
            g, str(exp_iri), measured_entity_class=tto.TTO_0000028,
            measurement_name='extensometer_nominal_gauge_length', numeric_value=ext_nom,
            unit_iri=unit.MilliM, bearer_iri=extensometer, bearer_predicate=RO_HAS_QUALITY,
            process_iri=process_iri, result_iri=result_iri,
            scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
            pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
            pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
            obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
            iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
        )

    # load cell force range
    lc_rng = row.get('Kraftmessdose-Kraftbereich')
    if lc_rng is not None and str(lc_rng).strip() != '' and str(lc_rng).lower() != 'nan':
        add_scalar_value_measurement(
            g, str(exp_iri), measured_entity_class=tto.TTO_0000023,
            measurement_name='load_cell_force_range', numeric_value=lc_rng,
            unit_iri=unit.KiloN, bearer_iri=load_cell, bearer_predicate=RO_HAS_QUALITY,
            process_iri=process_iri, result_iri=result_iri,
            scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
            pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
            pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
            obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
            iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
        )

    # Measurements
    for col, suffix in [('d1 / a1', 'a1'), ('d2 / a2', 'a2'), ('d3 / a3', 'a3')]:
        add_scalar_value_measurement(
            g, str(exp_iri), measured_entity_class=tto.TTO_0000029,
            measurement_name=f'original_thickness_{suffix}', numeric_value=row.get(col),
            unit_iri=unit.MilliM, bearer_iri=testpiece_iri, bearer_predicate=RO_HAS_QUALITY,
            process_iri=process_iri, result_iri=result_iri,
            scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
            pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
            pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
            obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
            iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
        )

    for col, suffix in [('b1', 'b1'), ('b2', 'b2'), ('b3', 'b3')]:
        add_scalar_value_measurement(
            g, str(exp_iri), measured_entity_class=tto.TTO_0000030,
            measurement_name=f'original_width_{suffix}', numeric_value=row.get(col),
            unit_iri=unit.MilliM, bearer_iri=testpiece_iri, bearer_predicate=RO_HAS_QUALITY,
            process_iri=process_iri, result_iri=result_iri,
            scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
            pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
            pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
            obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
            iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
        )

    for col, suffix in [('s1', 's1'), ('s2', 's2'), ('s3', 's3')]:
        add_scalar_value_measurement(
            g, str(exp_iri), measured_entity_class=tto.TTO_0000012,
            measurement_name=f'cross_section_area_{suffix}', numeric_value=row.get(col),
            unit_iri=unit.MilliM2, bearer_iri=testpiece_iri, bearer_predicate=RO_HAS_QUALITY,
            process_iri=process_iri, result_iri=result_iri,
            scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
            pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
            pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
            obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
            iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
        )

    add_scalar_value_measurement(
        g, str(exp_iri), measured_entity_class=tto.TTO_0000026,
        measurement_name='original_cross_section_area_S0', numeric_value=row.get('S0'),
        unit_iri=unit.MilliM2, bearer_iri=testpiece_iri, bearer_predicate=RO_HAS_QUALITY,
        process_iri=process_iri, result_iri=result_iri,
        scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
        pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
        obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
        iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
    )

    add_scalar_value_measurement(
        g, str(exp_iri), measured_entity_class=tto.TTO_0000028,
        measurement_name='original_gauge_length_L0', numeric_value=row.get('L0'),
        unit_iri=unit.MilliM, bearer_iri=testpiece_iri, bearer_predicate=RO_HAS_QUALITY,
        process_iri=process_iri, result_iri=result_iri,
        scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
        pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
        obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
        iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
    )

    add_scalar_value_measurement(
        g, str(exp_iri), measured_entity_class=tto.TTO_0000018,
        measurement_name='final_gauge_length_after_fracture_Lu', numeric_value=row.get('Lu'),
        unit_iri=unit.MilliM, bearer_iri=tp_after_1, bearer_predicate=RO_HAS_QUALITY,
        process_iri=process_iri, result_iri=result_iri,
        scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
        pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
        obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
        iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
    )

    add_scalar_value_measurement(
        g, str(exp_iri), measured_entity_class=tto.TTO_0000056,
        measurement_name='thickness_after_fracture_au', numeric_value=row.get('au'),
        unit_iri=unit.MilliM, bearer_iri=tp_after_1, bearer_predicate=RO_HAS_QUALITY,
        process_iri=process_iri, result_iri=result_iri,
        scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
        pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
        obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
        iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
    )

    add_scalar_value_measurement(
        g, str(exp_iri), measured_entity_class=tto.TTO_0000060,
        measurement_name='width_after_fracture_bu', numeric_value=row.get('bu'),
        unit_iri=unit.MilliM, bearer_iri=tp_after_1, bearer_predicate=RO_HAS_QUALITY,
        process_iri=process_iri, result_iri=result_iri,
        scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
        pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
        obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
        iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
    )

    add_scalar_value_measurement(
        g, str(exp_iri), measured_entity_class=tto.TTO_0000024,
        measurement_name='minimum_cross_section_area_after_fracture_Su', numeric_value=row.get('Su'),
        unit_iri=unit.MilliM2, bearer_iri=tp_after_1, bearer_predicate=RO_HAS_QUALITY,
        process_iri=process_iri, result_iri=result_iri,
        scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
        pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
        obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
        iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
    )
    
    # ------------------------------------------------------------------
    # du1 / du2 (diameter after fracture)
    # Column: du1, du2
    # TTO class: tto:TTO_0000015 (diameter after fracture)
    # bearer: fractured test piece part (tp_after_1)
    # unit: mm
    # ------------------------------------------------------------------

    add_scalar_value_measurement(
        g, str(exp_iri), measured_entity_class=tto.TTO_0000015,
        measurement_name='diameter_after_fracture_du1',
        numeric_value=row.get('du1'),
        unit_iri=unit.MilliM,
        bearer_iri=tp_after_1, bearer_predicate=RO_HAS_QUALITY,
        process_iri=process_iri, result_iri=result_iri,
        scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
        pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
        obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
        iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
    )

    add_scalar_value_measurement(
        g, str(exp_iri), measured_entity_class=tto.TTO_0000015,
        measurement_name='diameter_after_fracture_du2',
        numeric_value=row.get('du2'),
        unit_iri=unit.MilliM,
        bearer_iri=tp_after_1, bearer_predicate=RO_HAS_QUALITY,
        process_iri=process_iri, result_iri=result_iri,
        scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
        pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
        obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
        iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
    )

    # mechanical properties as qualities of material
    for (cls, name, col, unit_iri) in [
        (tto.TTO_0000047, 'proof_strength_Rp0_2', 'Rp0,2', unit.MegaPa),
        (tto.TTO_0000059, 'upper_yield_strength_ReH', 'ReH', unit.MegaPa),
        (tto.TTO_0000023, 'maximum_force_Fm', 'Fm', unit.KiloN),
        (tto.TTO_0000053, 'tensile_strength_Rm', 'Rm', unit.MegaPa),
        (ELASTIC_MODULUS, 'elastic_modulus_E', 'E', unit.GigaPa),
        (tto.TTO_0000033, 'percentage_elongation_after_fracture_A', 'A', unit.PERCENT),
        (tto.TTO_0000038, 'percentage_reduction_of_area_Z', 'Z', unit.PERCENT),
    ]:
        add_scalar_value_measurement(
            g, str(exp_iri), measured_entity_class=cls,
            measurement_name=name, numeric_value=row.get(col),
            unit_iri=unit_iri, bearer_iri=material_iri, bearer_predicate=RO_HAS_QUALITY,
            process_iri=process_iri, result_iri=result_iri,
            scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
            pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
            pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
            obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
            iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
        )

    # process attributes
    add_scalar_value_measurement(
        g, str(exp_iri), measured_entity_class=tto.TTO_0000051,
        measurement_name='strain_rate', numeric_value=row.get('Dehngeschwindigkeit'),
        unit_iri=None, bearer_iri=process_iri, bearer_predicate=PMD_HAS_PROCESS_ATTRIBUTE,
        process_iri=process_iri, result_iri=result_iri,
        scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
        pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
        obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
        iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
    )

    add_scalar_value_measurement(
        g, str(exp_iri), measured_entity_class=tto.TTO_0000058,
        measurement_name='transition_point_testing_rate', numeric_value=row.get('Umschaltpunkt'),
        unit_iri=unit.PERCENT, bearer_iri=process_iri, bearer_predicate=PMD_HAS_PROCESS_ATTRIBUTE,
        process_iri=process_iri, result_iri=result_iri,
        scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
        pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
        obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
        iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
    )

    add_scalar_value_measurement(
        g, str(exp_iri), measured_entity_class=ENV_TEMPERATURE,
        measurement_name='environmental_temperature', numeric_value=row.get('Umgebungstemperatur'),
        unit_iri=unit.DEG_C, bearer_iri=process_iri, bearer_predicate=PMD_HAS_PROCESS_ATTRIBUTE,
        process_iri=process_iri, result_iri=result_iri,
        scalar_value_spec_class=PMD_SVS, scalar_measurement_datum_class=PMD_SMD,
        pmd_has_value=PMD_HAS_VALUE, pmd_has_unit=PMD_HAS_UNIT,
        pmd_has_value_spec=PMD_HAS_VALUE_SPEC, pmd_specifies_value_of=PMD_SPECIFIES_VALUE_OF,
        obi_has_specified_output=OBI_HAS_SPECIFIED_OUTPUT, bfo_has_part=BFO_HAS_PART,
        iao_is_about=IAO_IS_ABOUT, iao_is_quality_measured_as=IAO_IS_QUALITY_MEASURED_AS
    )

    # operator + date (if present)    
    operator_name = row.get('Operator')
    if operator_name is not None and str(operator_name).strip() != '' and str(operator_name).lower() != 'nan':
        operator_iri = URIRef(str(exp_iri) + '_operator')
        g.add((operator_iri, a, obo.BFO_0000040))  # since there is no "operator" class in BFO and PMDco, and "person" is not imported, "material entity" is instantiated for now
        g.add((operator_iri, RDFS.label, Literal(str(operator_name), datatype=XSD.string)))

        operator_role_iri = URIRef(str(operator_iri) + '_investigation_agent_role')
        g.add((operator_role_iri, a, obo.OBI_0000202))  # "investigation agent role" from OBI is instantiated as role for the operator (since there is no "operator role" class in RO)
        g.add((operator_iri, obo.RO_0000087, operator_role_iri))  # has role

        g.add((process_iri, RO_HAS_PARTICIPANT, operator_iri))


    date_val = row.get('date')
    if date_val is not None and str(date_val).strip() != '' and str(date_val).lower() != 'nan':
        date_iri = URIRef(str(exp_iri) + '_date')
        g.add((date_iri, a, time.Instant))
        g.add((date_iri, time.inXSDDate, Literal(str(date_val), datatype=XSD.date)))
        g.add((process_iri, obo.BFO_0000108, date_iri))
        
    # ------------------------------------------------------------------
    # Note (string) as identifier attached to the process
    # Column: Note
    # ------------------------------------------------------------------
    note_val = row.get('Note')
    if note_val is not None and str(note_val).strip() != '' and str(note_val).lower() != 'nan':
        add_identifier(
            g, base_iri=str(process_iri), thing_iri=process_iri, value=note_val,
            id_suffix='note', identifier_class=obo.IAO_0020000,
            value_spec_class=obo.OBI_0001933, pmd_has_value=PMD_HAS_VALUE,
            pmd_has_value_spec=PMD_HAS_VALUE_SPEC, iao_denotes=obo.IAO_0000219,
            label='process note'
        )
    
print('Number of triples created, that are now to be found in the graph: ', len(g))

# show potential unmapped columns (completeness check)
unmapped = [c for c in data.columns if c not in USED_COLS]
print('Mandatory completeness check: Unmapped CSV columns (not yet used in mapping):')
for c in unmapped:
    print(' -', c)


Number of triples created, that are now to be found in the graph:  6876
Mandatory completeness check: Unmapped CSV columns (not yet used in mapping):


# SPARQL ASK Checks (Consistency checks) + Debug SELECT helpers

The following cell contains a set of **ASK queries** for a quick consistency validation of the generated A-Box.

Among other things, it checks:
- **Scalar Value Measurement Pattern**: Every `pmdco:PMD_0000023` (scalar measurement datum) has exactly one value specification via `pmdco:PMD_0060000`, and that value specification contains a value via `pmdco:PMD_0000006`.
- **Result aggregation**: Every tensile testing process has a `tto:tensile test result` as a specified output, and the result contains measurement data items as parts.
- **Device/Function pattern**: Devices bear functions, and those functions are realized in the process.

> Tip: If an ASK check returns `False`, corresponding WHERE patterns can be rewritten as SELECT queries to identify the problematic IRIs. This is performed for some of the ASK checks as examples.

In [ ]:
# RDFlib ASK checks with selected debug SELECT outputs

def ask(label, q):
    r = g.query(q)
    ans = getattr(r, 'askAnswer', None)
    ok = bool(ans) if ans is not None else bool(r)
    print(f"{label}: {ok}")
    return ok

def select(label, q, limit=20):
    rows = list(g.query(q))
    print(f"{label}: {len(rows)} rows")
    for r in rows[:limit]:
        print(' ', r)
    if len(rows) > limit:
        print(f"  ... (showing first {limit})")
    return rows

PREFIXES = """PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>PREFIX obo: <http://purl.obolibrary.org/obo/>PREFIX pmdco: <https://w3id.org/pmd/co/>PREFIX tto: <https://w3id.org/pmd/tto/>PREFIX csvw: <http://www.w3.org/ns/csvw#>"""

checks = {}

# 1) Every scalar measurement datum has exactly one scalar value specification via PMD_0060000
ask(
    "1) Every SMD has exactly one SVS (pmdco:PMD_0060000)",
    PREFIXES + """ASK {  FILTER NOT EXISTS {    SELECT ?smd (COUNT(?svs) AS ?c)    WHERE {      ?smd a pmdco:PMD_0000023 .      OPTIONAL { ?smd pmdco:PMD_0060000 ?svs . }    }    GROUP BY ?smd    HAVING (?c != 1)  }}"""
)

# 2) Every SVS has a value and specifies a measured entity
ask(
    "2) Every SVS has PMD_0000006 (value) and PMD_0060001 (specifies value of)",
    PREFIXES + """ASK {  FILTER NOT EXISTS {    ?svs a pmdco:PMD_0000022 .    FILTER NOT EXISTS { ?svs pmdco:PMD_0000006 ?v . }    FILTER NOT EXISTS { ?svs pmdco:PMD_0060001 ?measured . }  }}"""
)

# 3) Every measured entity referenced by SVS is linked to an SMD via IAO_0000417
ask(
    "3) Every measured entity used in SVS is linked to an SMD (obo:IAO_0000417)",
    PREFIXES + """ASK {  FILTER NOT EXISTS {    ?svs a pmdco:PMD_0000022 ; pmdco:PMD_0060001 ?measured .    FILTER NOT EXISTS { ?measured obo:IAO_0000417 ?smd . }  }}"""
)

# 4) Every tensile testing process has a tensile test result as specified output
ask(
    "4) Every tensile test process has a tensile test result as specified output",
    PREFIXES + """ASK {  FILTER NOT EXISTS {    ?p a pmdco:PMD_0000974 .    FILTER NOT EXISTS {      ?p obo:OBI_0000299 ?res .      ?res a tto:TTO_0000008 .    }  }}"""
)

# 5) Every tensile test result has at least one SMD as part
ask(
    "5) Every tensile test result has at least one SMD (pmdco:PMD_0000023) as part",
    PREFIXES + """ASK {  FILTER NOT EXISTS {    ?res a tto:TTO_0000008 .    FILTER NOT EXISTS { ?res obo:BFO_0000051 ?smd . ?smd a pmdco:PMD_0000023 . }  }}"""
)

# 6) Devices bear a function and the function is realized in the tensile test process
ask(
    "6) Devices bear a function and the function is realized in the tensile test process",
    PREFIXES + """ASK {  FILTER NOT EXISTS {    ?p a pmdco:PMD_0000974 ; obo:RO_0000057 ?dev .    FILTER(STRSTARTS(STR(?dev), "https://w3id.org/pmd/demodata/tensiletest_S355/"))    FILTER NOT EXISTS {      ?dev obo:BFO_0000196 ?f .      ?f obo:BFO_0000054 ?p .    }  }}"""
)

# 7) If an institution node exists, it has at least one identifier value
ask(
    "7) Institution has at least one identifier value",
    PREFIXES + """ASK {  FILTER NOT EXISTS {    ?org a obo:OBI_0000245 .    FILTER(CONTAINS(STR(?org), "_institution"))    FILTER NOT EXISTS {      ?id a obo:IAO_0020000 ; obo:IAO_0000219 ?org ; pmdco:PMD_0060000 ?vs .      ?vs pmdco:PMD_0000006 ?val .    }  }}"""
)

# 8) Norm
q8 = PREFIXES + """ASK {  FILTER NOT EXISTS {    ?p a pmdco:PMD_0000974 .    FILTER NOT EXISTS {      ?p obo:BFO_0000055 ?plan .      ?plan a tto:TTO_0000054 .      ?id a obo:IAO_0020000 ; obo:IAO_0000219 ?plan ; pmdco:PMD_0060000 ?vs .      ?vs pmdco:PMD_0000006 ?std .    }  }}"""
checks['8'] = ask('8) Norm present as plan + identifier', q8)
if not checks['8']:
    select('DEBUG 8) processes missing plan/standard', PREFIXES + """SELECT ?p WHERE {  ?p a pmdco:PMD_0000974 .  FILTER NOT EXISTS {    ?p obo:BFO_0000055 ?plan .    ?plan a tto:TTO_0000054 .    ?id a obo:IAO_0020000 ; obo:IAO_0000219 ?plan ; pmdco:PMD_0060000 ?vs .    ?vs pmdco:PMD_0000006 ?std .  }}""")

# 9) Project
q9 = PREFIXES + """ASK {  FILTER NOT EXISTS {    ?p a pmdco:PMD_0000974 .    FILTER NOT EXISTS {      ?proj a pmdco:PMD_0000909 ; obo:BFO_0000117 ?p .      ?id a obo:IAO_0020000 ; obo:IAO_0000219 ?proj ; pmdco:PMD_0060000 ?vs .      ?vs pmdco:PMD_0000006 ?pid .    }  }}"""
checks['9'] = ask('9) Project present with identifier', q9)
if not checks['9']:
    select('DEBUG 9) processes missing project/identifier', PREFIXES + """SELECT ?p WHERE {  ?p a pmdco:PMD_0000974 .  FILTER NOT EXISTS {    ?proj a pmdco:PMD_0000909 ; obo:BFO_0000117 ?p .    ?id a obo:IAO_0020000 ; obo:IAO_0000219 ?proj ; pmdco:PMD_0060000 ?vs .    ?vs pmdco:PMD_0000006 ?pid .  }}""")

# 10) Rolling direction
q10 = PREFIXES + """ASK {  FILTER NOT EXISTS {    ?m a pmdco:PMD_0020096 .    FILTER NOT EXISTS {      ?m obo:RO_0000086 ?q .      FILTER(CONTAINS(STR(?q), "rolling_direction"))      ?q pmdco:PMD_0060000 ?vs .      ?vs pmdco:PMD_0000006 ?val .    }  }}"""
checks['10'] = ask('10) Rolling direction quality present', q10)
if not checks['10']:
    select('DEBUG 10) materials missing rolling direction', PREFIXES + """SELECT ?m WHERE {  ?m a pmdco:PMD_0020096 .  FILTER NOT EXISTS {    ?m obo:RO_0000086 ?q .    FILTER(CONTAINS(STR(?q), "rolling_direction"))    ?q pmdco:PMD_0060000 ?vs .    ?vs pmdco:PMD_0000006 ?val .  }}""")

# CSVW presence check
qcsvw = PREFIXES + """ASK {  ?t a csvw:Table ; csvw:url ?u ; csvw:tableSchema ?s .}"""
checks['csvw'] = ask('CSVW table description present', qcsvw)


1) Every SMD has exactly one SVS (pmdco:PMD_0060000): True
2) Every SVS has PMD_0000006 (value) and PMD_0060001 (specifies value of): True
3) Every measured entity used in SVS is linked to an SMD (obo:IAO_0000417): True
4) Every tensile test process has a tensile test result as specified output: True
5) Every tensile test result has at least one SMD (pmdco:PMD_0000023) as part: True
6) Devices bear a function and the function is realized in the tensile test process: True
7) Institution has at least one identifier value: True
8) Norm present as plan + identifier: True
9) Project present with identifier: True
10) Rolling direction quality present: True
CSVW table description present: True


# Graph serialization

In [9]:
out_ttl = 'S355_data_tto_pmdco3.ttl'
out_rdf = 'S355_data_tto_pmdco3.rdf'

# Graph g is serialized in two different data formats / notations, Turtle (ttl) and RDF
g.serialize(out_ttl, format='ttl')
g.serialize(out_rdf, format='application/rdf+xml', encoding='utf-8')

# Instead of serializing the data to be stored in the working directory, 
# the files could alternatively be stored in another repository or directly
# uploaded to a triple store.

<Graph identifier=N0c0511c54db842ce83cc38325c3fa653 (<class 'rdflib.graph.Graph'>)>